In [ ]:

Este notebook carga y analiza los datos de productos de maquillaje, ajusta las rutas de imagen automáticamente y genera visualizaciones clave del catálogo.</VSCode.Cell>
<VSCode.Cell language="python">import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid', font_scale=1.1)

base_path = Path(r'c:\Users\LENOVO\OneDrive\Desktop\SAEN PRO')
file_path = base_path / 'datos-maquillaje.js'

raw_text = file_path.read_text(encoding='utf-8')
json_text = raw_text.split('const PRODS_MAQUILLAJE =', 1)[1].rsplit(';', 1)[0].strip()
prods_maquillaje = json.loads(json_text)

df = pd.json_normalize(prods_maquillaje)
df.head()</VSCode.Cell>
<VSCode.Cell language="markdown">## 2. Limpiar y Estructurar el DataFrame

Validar tipos de datos, detectar valores faltantes y expandir la estructura de precios para análisis individual.</VSCode.Cell>
<VSCode.Cell language="python">price_df = df.explode('precios').reset_index(drop=True)
price_df = pd.concat([price_df.drop(columns=['precios']), price_df['precios'].apply(pd.Series)], axis=1)

price_df['valor'] = pd.to_numeric(price_df['valor'], errors='coerce')
price_df['tipo'] = price_df['tipo'].astype(str).str.strip().str.upper()

cleaning_report = {
    'dtypes': df.dtypes,
    'missing_values': df.isna().sum(),
    'price_types': price_df['tipo'].unique()
}
cleaning_report</VSCode.Cell>
<VSCode.Cell language="markdown">## 3. Análisis Estadístico de Precios

Calcular estadísticas descriptivas por tipo de precio y visualizar distribuciones.</VSCode.Cell>
<VSCode.Cell language="python">price_stats = price_df.groupby('tipo')['valor'].describe().round(2)
price_stats

plt.figure(figsize=(10, 6))
sns.boxplot(data=price_df, x='tipo', y='valor')
plt.title('Distribución de Precios por Tipo')
plt.xlabel('Tipo de precio')
plt.ylabel('Valor')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(data=price_df, x='valor', hue='tipo', multiple='stack', bins=30)
plt.title('Histograma de Precios por Tipo')
plt.xlabel('Valor')
plt.tight_layout()
plt.show()</VSCode.Cell>
<VSCode.Cell language="markdown">## 4. Visualizar Distribución de Productos por Badge y Estado

Crear gráficos para mostrar la distribución de productos según badge y disponibilidad.</VSCode.Cell>
<VSCode.Cell language="python">badge_counts = df['badge'].replace('', 'Sin badge').value_counts()
status_counts = df['estado'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(x=badge_counts.index, y=badge_counts.values, ax=axes[0])
axes[0].set_title('Distribución de Badges')
axes[0].set_xlabel('Badge')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x=status_counts.index, y=status_counts.values, ax=axes[1])
axes[1].set_title('Estado de Disponibilidad')
axes[1].set_xlabel('Estado')
axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()</VSCode.Cell>
<VSCode.Cell language="markdown">## 5. Análisis de Calificaciones y Reseñas

Analizar la relación entre estrellas y número de reseñas, y encontrar productos destacados.</VSCode.Cell>
<VSCode.Cell language="python">df['estrellas'] = pd.to_numeric(df['estrellas'], errors='coerce')
df['resenas'] = pd.to_numeric(df['resenas'], errors='coerce')

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='resenas', y='estrellas', hue='badge', alpha=0.8)
plt.title('Relación entre Reseñas y Estrellas')
plt.xlabel('Número de reseñas')
plt.ylabel('Estrellas')
plt.legend(title='Badge')
plt.tight_layout()
plt.show()

correlation = df[['resenas', 'estrellas']].corr().iloc[0, 1]
correlation</VSCode.Cell>
<VSCode.Cell language="markdown">## 6. Estrategia de Precios por Tipo

Comparar el ahorro en diferentes tipos de compra y visualizar tendencias.</VSCode.Cell>
<VSCode.Cell language="python">discount_df = price_df.pivot_table(index='id', columns='tipo', values='valor', aggfunc='first')

if 'UNIDAD' in discount_df.columns and 'MAYOR' in discount_df.columns:
    discount_df['ahorro_mayor_pct'] = 100 * (discount_df['UNIDAD'] - discount_df['MAYOR']) / discount_df['UNIDAD']
    discount_pct = discount_df['ahorro_mayor_pct'].dropna().describe().round(2)
else:
    discount_pct = pd.Series(dtype=float)

discount_pct</VSCode.Cell>
<VSCode.Cell language="markdown">## 7. Generar Rutas de Imágenes Automáticamente

Crear una función que genere la ruta `IMAGENES MAQUILLAJE/IMGMAQ###.png` según el ID del producto.</VSCode.Cell>
<VSCode.Cell language="python">def generar_ruta_imagen(id_producto: str) -> str:
    numero = ''.join(filter(str.isdigit, id_producto))
    return f'IMAGENES MAQUILLAJE/IMGMAQ{int(numero):03}.png'

rutas_validas = [generar_ruta_imagen(pid) for pid in df['id'].head(10)]
rutas_validas</VSCode.Cell>
<VSCode.Cell language="markdown">## 8. Crear Dashboard Interactivo de Productos

Preparar un dashboard interactivo simple con filtros y tablas. Este notebook muestra cómo organizar los datos y visualizaciones.</VSCode.Cell>
<VSCode.Cell language="python">import ipywidgets as widgets
from IPython.display import display

precio_min = int(price_df['valor'].min())
precio_max = int(price_df['valor'].max())

slider_precio = widgets.IntRangeSlider(
    value=[precio_min, precio_max],
    min=precio_min,
    max=precio_max,
    step=1,
    description='Rango precio:',
    continuous_update=False
)

selector_estado = widgets.Dropdown(
    options=['todos'] + df['estado'].dropna().unique().tolist(),
    value='todos',
    description='Estado:'
)

output = widgets.Output()

@widgets.interact

def dashboard(rango=slider_precio, estado=selector_estado):
    with output:
        output.clear_output()
        filtro = df.copy()
        if estado != 'todos':
            filtro = filtro[filtro['estado'] == estado]
        filtro = filtro[filtro['id'].isin(price_df[price_df['valor'].between(rango[0], rango[1])]['id'])]
        display(filtro[['id', 'nombre', 'categoria', 'badge', 'estado', 'estrellas', 'resenas']].head(20))

    display(output)